# Preprocessing — ViFactCheck to COOLANT HDF5

In [ ]:
import os
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

# DATA_ROOT: where large files live (json, jpg, hdf5, processed_data).
# Defaults to DATA_ROOT env var, or PROJECT_ROOT if unset.
DATA_ROOT = Path(os.environ["DATA_ROOT"]) if os.environ.get("DATA_ROOT") else PROJECT_ROOT

DATA_SOURCE = "vifactcheck"
LABEL_VARIANT = "root"
SPLITS = ["train", "dev", "test"]

VIFACTCHECK_JSON_DIR = DATA_ROOT / "data" / "json"
CRAWLED_JSON_DIR = DATA_ROOT / "data" / "json"
JPG_DIR = DATA_ROOT / "data" / "jpg"
OUTPUT_DIR = DATA_ROOT / "processed_data" / "hdf5"
PAIRS_CACHE_DIR = OUTPUT_DIR / "pairs"

AUTO_INSTALL_DEPS = False
FORCE_REBUILD = False

TEXT_MODEL_NAME = "vinai/phobert-base-v2"
IMAGE_MODEL_NAME = "resnet50"
MAX_LENGTH = 128
BATCH_SIZE = 32
MAX_PAIRS_PER_SPLIT = None

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DATA_ROOT:    {DATA_ROOT}")
print(f"DATA_SOURCE: {DATA_SOURCE}")
print(f"LABEL_VARIANT: {LABEL_VARIANT}")

In [ ]:
import importlib

required_packages = {
    "h5py": "h5py",
    "torch": "torch",
    "transformers": "transformers",
    "torchvision": "torchvision",
    "PIL": "Pillow",
    "numpy": "numpy",
    "pandas": "pandas",
    "tqdm": "tqdm",
}

missing_packages = []
for import_name, package_name in required_packages.items():
    try:
        importlib.import_module(import_name)
    except ImportError:
        missing_packages.append((import_name, package_name))

if missing_packages:
    print("\n" + "="*60)
    print("MISSING DEPENDENCIES")
    print("="*60)
    for import_name, package_name in missing_packages:
        print(f"  - {package_name}")
    
    print("\nTo install, run:")
    print("  conda install -n fake_news h5py -c conda-forge")
    print("  pip install h5py")
    print("\nOr set AUTO_INSTALL_DEPS=True in the config cell above.")
    print("="*60)
    
    if not AUTO_INSTALL_DEPS:
        raise RuntimeError(f"Missing dependencies: {[p[0] for p in missing_packages]}. Install them and re-run.")
else:
    print("✓ All required dependencies are installed.")

In [ ]:
import gc
import json
import h5py
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Add src/ so bare `import preprocessing` and `import utils` resolve correctly
_src = str(PROJECT_ROOT / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PAIRS_CACHE_DIR.mkdir(parents=True, exist_ok=True)

from processing.coolant.pair_extractor import PairExtractor

print("✓ Setup complete. Ready for preprocessing.")

## Step 1: Resolve Input Files

In [ ]:
def resolve_json_path(split, data_source, label_variant):
    """Resolve JSON path based on data source and label variant."""
    if data_source == "vifactcheck":
        if label_variant == "root":
            return VIFACTCHECK_JSON_DIR / f"news_data_vifactcheck_{split}_labeled.json"
        elif label_variant == "nei_as_real":
            return VIFACTCHECK_JSON_DIR / "labeled_nei_as_real" / f"news_data_vifactcheck_{split}_labeled.json"
        elif label_variant == "three_class":
            return VIFACTCHECK_JSON_DIR / "labeled_three_class" / f"news_data_vifactcheck_{split}_labeled.json"
    elif data_source == "crawled":
        return CRAWLED_JSON_DIR / f"news_data_{split}.json"
    elif data_source == "merged":
        if split in ["dev", "test"]:
            return VIFACTCHECK_JSON_DIR / f"news_data_vifactcheck_{split}_labeled.json"
        else:
            return CRAWLED_JSON_DIR / f"news_data_{split}.json"
    
    raise ValueError(f"Unknown data_source: {data_source}")

def load_articles(path):
    """Load articles from JSON file."""
    if not path.exists():
        print(f"  ⚠ File not found: {path}")
        return []
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

print("✓ Resolver functions defined.")

## Step 2: Extract Valid Matched Pairs

In [ ]:
extractor = PairExtractor(jpg_base_dir=str(JPG_DIR), min_caption_len=5)

all_pairs = {}
stats = {}

for split in SPLITS:
    json_path = resolve_json_path(split, DATA_SOURCE, LABEL_VARIANT)
    articles = load_articles(json_path)
    
    if not articles:
        print(f"  {split}: skipped (no data)")
        continue
    
    print(f"\n{split.upper()}:")
    print(f"  Raw articles: {len(articles)}")
    
    pairs, pair_stats = extractor.extract_from_json(str(json_path), return_stats=True)
    
    if MAX_PAIRS_PER_SPLIT is not None:
        pairs = pairs[:MAX_PAIRS_PER_SPLIT]
        print(f"  Truncated to {len(pairs)} pairs (MAX_PAIRS_PER_SPLIT={MAX_PAIRS_PER_SPLIT})")
    
    all_pairs[split] = pairs
    stats[split] = pair_stats
    
    cache_path = PAIRS_CACHE_DIR / f"pairs_{split}.json"
    with open(cache_path, "w", encoding="utf-8") as f:
        json.dump(pairs, f, ensure_ascii=False, indent=2)
    print(f"  Cached to {cache_path.name}")

print("\n✓ Pair extraction complete.")

## Step 3: Initialize Feature Extractors

In [ ]:
def get_device():
    """Auto-detect device: cuda > mps > cpu."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif torch.backends.mps.is_available():
        return torch.device("mps")
    else:
        return torch.device("cpu")

device = get_device()
print(f"Device: {device}")

from preprocessing.text_preprocessing import TextPreprocessor
from preprocessing.image_preprocessing import ImagePreprocessor

text_preprocessor = TextPreprocessor(
    model_name=TEXT_MODEL_NAME,
    max_length=MAX_LENGTH,
    language="vi",
    device=str(device),
    use_word_segmentation=True
)

image_preprocessor = ImagePreprocessor(
    model_name=IMAGE_MODEL_NAME,
    device=str(device)
)

print("✓ Feature extractors initialized.")

## Step 4: Extract Features and Save HDF5

In [ ]:
for split in SPLITS:
    if split not in all_pairs:
        print(f"\n{split.upper()}: skipped (no pairs)")
        continue
    
    pairs = all_pairs[split]
    output_file = OUTPUT_DIR / f"coolant_{split}.h5"
    
    if output_file.exists() and not FORCE_REBUILD:
        print(f"\n{split.upper()}: {output_file.name} exists, skipping (set FORCE_REBUILD=True to overwrite)")
        continue
    
    print(f"\n{split.upper()}: Extracting features for {len(pairs)} pairs...")
    
    caption_features_list = []
    image_features_list = []
    article_ids_list = []
    source_labels_list = []
    source_urls_list = []
    image_paths_list = []
    folder_paths_list = []
    
    for i in tqdm(range(0, len(pairs), BATCH_SIZE), desc=f"  Batches"):
        batch_pairs = pairs[i:i+BATCH_SIZE]
        
        batch_texts = [p.get("pair_text") or f"{p.get('title', '')} {p.get('caption', '')}".strip() for p in batch_pairs]
        batch_image_paths = [p["image_path"] for p in batch_pairs]
        
        text_features = text_preprocessor.extract_token_embeddings(batch_texts)
        image_features = image_preprocessor.extract_features(batch_image_paths)
        
        caption_features_list.append(text_features)
        image_features_list.append(image_features)
        
        for p in batch_pairs:
            article_ids_list.append(str(p.get("article_idx", "")))
            source_labels_list.append(p.get("source_label", ""))
            source_urls_list.append(p.get("source_url", ""))
            image_paths_list.append(p.get("image_path", ""))
            folder_paths_list.append(p.get("folder_path", ""))
        
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    
    caption_features = np.vstack(caption_features_list)
    image_features = np.vstack(image_features_list)
    
    with h5py.File(output_file, "w") as f:
        f.create_dataset("caption_features", data=caption_features)
        f.create_dataset("image_features", data=image_features)
        f.create_dataset("article_ids", data=np.array(article_ids_list, dtype="S"))
        f.create_dataset("source_labels", data=np.array(source_labels_list, dtype="S"))
        f.create_dataset("source_urls", data=np.array(source_urls_list, dtype="S"))
        f.create_dataset("image_paths", data=np.array(image_paths_list, dtype="S"))
        f.create_dataset("folder_paths", data=np.array(folder_paths_list, dtype="S"))
        
        f.attrs["n_samples"] = len(pairs)
        f.attrs["caption_shape"] = caption_features.shape
        f.attrs["image_shape"] = image_features.shape
        f.attrs["text_model"] = TEXT_MODEL_NAME
        f.attrs["image_model"] = IMAGE_MODEL_NAME
        f.attrs["max_length"] = MAX_LENGTH
        f.attrs["data_source"] = DATA_SOURCE
        f.attrs["label_variant"] = LABEL_VARIANT
    
    print(f"  Saved to {output_file.name}")
    print(f"    Caption features: {caption_features.shape}")
    print(f"    Image features: {image_features.shape}")

print("\n✓ HDF5 extraction complete.")

## Step 5: Dataset Statistics Report

In [ ]:
report_data = []

for split in SPLITS:
    if split not in stats:
        continue
    
    s = stats[split]
    output_file = OUTPUT_DIR / f"coolant_{split}.h5"
    
    hdf5_rows = 0
    caption_shape = None
    image_shape = None
    file_size_mb = 0
    
    if output_file.exists():
        with h5py.File(output_file, "r") as f:
            hdf5_rows = f.attrs.get("n_samples", 0)
            caption_shape = f.attrs.get("caption_shape", "")
            image_shape = f.attrs.get("image_shape", "")
        file_size_mb = output_file.stat().st_size / (1024 * 1024)
    
    report_data.append({
        "split": split,
        "raw_articles": s["raw_articles"],
        "valid_pairs": s["valid_pairs"],
        "skipped_no_image": s["skipped"].get("no_image", 0),
        "skipped_no_caption": s["skipped"].get("no_caption", 0),
        "skipped_credit_only": s["skipped"].get("credit_only", 0),
        "skipped_too_short": s["skipped"].get("too_short", 0),
        "hdf5_rows": hdf5_rows,
        "source_label_counts": s["source_label_counts"],
        "caption_shape": str(caption_shape),
        "image_shape": str(image_shape),
        "file_size_mb": f"{file_size_mb:.2f}"
    })

df_report = pd.DataFrame(report_data)
print("\nDataset Statistics:")
print(df_report.to_string(index=False))

stats_file = OUTPUT_DIR / "preprocessing_stats.json"
with open(stats_file, "w", encoding="utf-8") as f:
    json.dump(report_data, f, ensure_ascii=False, indent=2)
print(f"\n✓ Stats saved to {stats_file.name}")

## Step 6: Verify COOLANT Dataset Loading

In [ ]:
print("Verifying HDF5 files...\n")

for split in SPLITS:
    output_file = OUTPUT_DIR / f"coolant_{split}.h5"
    if not output_file.exists():
        print(f"{split}: file not found")
        continue
    
    with h5py.File(output_file, "r") as f:
        required_datasets = ["caption_features", "image_features", "article_ids"]
        for ds in required_datasets:
            assert ds in f, f"Missing dataset: {ds}"
        
        n_samples = f.attrs["n_samples"]
        caption_shape = f["caption_features"].shape
        image_shape = f["image_features"].shape
        
        print(f"{split}:")
        print(f"  Samples: {n_samples}")
        print(f"  Caption shape: {caption_shape}")
        print(f"  Image shape: {image_shape}")

print("\n✓ Ready for Phase 3: COOLANT Training Notebook")